# ADAUSDT trade distribution features

???????? ?????? ????????? minute-level feature dataset ????????? ????????????? ??????? ?????? ?? ?????? raw `trades`, ?? ??????? raw-????. Minute backbone ??????? ?? raw `klines` ? ?????? ?????? ?????? ????? ? ???????? parquet.

In [1]:
import io
import os
import re

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

BUCKET = "binance-data-downloader"
RAW_PREFIX = "raw"
FEATURES_PREFIX = "features"
FEATURE_DATASET_NAME = "trade_distribution"
SYMBOL = "ADAUSDT"
INTERVAL = "1m"
SKIP_EXISTING = True
TOP_FRACTION = 0.2

FEATURE_COLUMNS = [
    "open_time",
    "Cp_buy",
    "Cp_sell",
    "H_buy",
    "H_sell",
    "H_norm_buy",
    "H_norm_sell",
    "N_eff_buy",
    "N_eff_sell",
    "C_eff_buy",
    "C_eff_sell",
]
FLOAT_FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c != "open_time"]


def make_s3_client():
    load_dotenv()
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )

In [2]:
def list_symbol_trades_days(symbol: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, s3_client=None) -> list[str]:
    s3=s3_client or make_s3_client()
    source_prefix=f"{raw_prefix.strip('/')}/trades/symbol={symbol}/"
    pattern=re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")
    dates=set()
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            m=pattern.search('/'+obj['Key'])
            if m:
                dates.add(m.group(1))
    return sorted(dates)


def trades_key(symbol: str, date: str, raw_prefix: str = RAW_PREFIX) -> str:
    return f"{raw_prefix.strip('/')}/trades/symbol={symbol}/date={date}/data.parquet"


def klines_key(symbol: str, date: str, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL) -> str:
    return f"{raw_prefix.strip('/')}/klines/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def feature_dataset_key(symbol: str, date: str, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL) -> str:
    return f"{features_prefix.strip('/')}/{FEATURE_DATASET_NAME}/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def s3_key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except Exception as exc:
        code=getattr(exc,"response",{}).get("Error",{}).get("Code")
        if code in {"404","NoSuchKey","NotFound"}:
            return False
        raise

In [3]:
def read_symbol_trades_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, s3_client=None) -> pd.DataFrame:
    s3=s3_client or make_s3_client()
    obj=s3.get_object(Bucket=bucket, Key=trades_key(symbol,date,raw_prefix))
    df=pd.read_parquet(io.BytesIO(obj["Body"].read()))
    required=["time","qty","is_buyer_maker"]
    missing=sorted(set(required)-set(df.columns))
    if missing:
        raise ValueError(f"Missing required trades columns: {missing}")
    if df.empty:
        return pd.DataFrame(columns=required)
    out=df[required].copy()
    out["time"]=pd.to_numeric(out["time"], errors="coerce").astype("Int64")
    out["qty"]=pd.to_numeric(out["qty"], errors="coerce")
    out["is_buyer_maker"]=out["is_buyer_maker"].astype("boolean")
    return out.dropna(subset=required).reset_index(drop=True)


def read_minute_backbone_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL, s3_client=None) -> pd.DataFrame:
    s3=s3_client or make_s3_client()
    obj=s3.get_object(Bucket=bucket, Key=klines_key(symbol,date,raw_prefix,interval))
    df=pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=["open_time"])
    if df.empty:
        return pd.DataFrame(columns=["open_time"])
    out=df[["open_time"]].copy()
    out["open_time"]=pd.to_numeric(out["open_time"], errors="coerce").astype("Int64")
    out=out.dropna(subset=["open_time"]).drop_duplicates(subset=["open_time"]).sort_values("open_time").reset_index(drop=True)
    ts=pd.to_datetime(out["open_time"].astype("int64"), unit="ms", utc=True)
    if not ts.dt.second.eq(0).all() or not ts.dt.microsecond.eq(0).all():
        raise ValueError("Minute backbone open_time must be minute-aligned UTC")
    return out

In [4]:
def summarize_distribution(qty: pd.Series, top_fraction: float = TOP_FRACTION) -> pd.Series:
    values = qty.astype("float64").to_numpy()
    n = len(values)
    if n == 0:
        return pd.Series({"Cp": 0.0, "H": 0.0, "H_norm": 0.0, "N_eff": 0.0, "C_eff": 0.0})
    total = values.sum()
    if total <= 0:
        return pd.Series({"Cp": 0.0, "H": 0.0, "H_norm": 0.0, "N_eff": 0.0, "C_eff": 0.0})
    k = max(1, int(np.floor(top_fraction * n)))
    sorted_desc = np.sort(values)[::-1]
    cp = sorted_desc[:k].sum() / total
    p = values / total
    if n <= 1:
        h = 0.0
        h_norm = 0.0
    else:
        h = float(-(p * np.log(p)).sum())
        h_norm = float(h / np.log(n))
    n_eff = float(1.0 / np.sum(p ** 2))
    c_eff = float(n_eff / n)
    return pd.Series({"Cp": cp, "H": h, "H_norm": h_norm, "N_eff": n_eff, "C_eff": c_eff})


def build_trade_distribution_for_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL, s3_client=None) -> pd.DataFrame:
    backbone=read_minute_backbone_day(symbol,date,bucket,raw_prefix,interval,s3_client)
    trades=read_symbol_trades_day(symbol,date,bucket,raw_prefix,s3_client)
    if backbone.empty:
        return pd.DataFrame(columns=FEATURE_COLUMNS)
    if trades.empty:
        out=backbone.copy()
        for c in FLOAT_FEATURE_COLUMNS:
            out[c]=np.float32(0)
        return out[FEATURE_COLUMNS]
    trades=trades.assign(
        open_time=((trades["time"].astype("int64")//60000)*60000).astype("int64"),
        side=np.where(~trades["is_buyer_maker"], "buy", "sell"),
    )
    grouped_rows = []
    for (open_time, side), side_qty in trades.groupby(["open_time", "side"])["qty"]:
        stats = summarize_distribution(side_qty)
        grouped_rows.append({"open_time": open_time, "side": side, **stats.to_dict()})

    summary_long = pd.DataFrame(grouped_rows)
    summary = summary_long.pivot(index="open_time", columns="side", values=["Cp", "H", "H_norm", "N_eff", "C_eff"])
    summary.columns = [f"{metric}_{side}" for metric, side in summary.columns]
    summary = summary.reset_index()
    out=backbone.merge(summary,on="open_time",how="left").fillna(0)
    for c in FLOAT_FEATURE_COLUMNS:
        if c not in out.columns:
            out[c]=0
        out[c]=out[c].astype("float32")
    return out[FEATURE_COLUMNS].sort_values("open_time").reset_index(drop=True)

In [5]:
def write_trade_distribution_for_symbol_to_s3(symbol: str = SYMBOL, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL, skip_existing: bool = SKIP_EXISTING, s3_client=None) -> pd.DataFrame:
    s3=s3_client or make_s3_client()
    dates=list_symbol_trades_days(symbol,bucket,raw_prefix,s3)
    if not dates:
        raise FileNotFoundError(f"No trades days found for symbol={symbol}")
    rows=[]
    for date in dates:
        key=feature_dataset_key(symbol,date,features_prefix,interval)
        if skip_existing and s3_key_exists(s3,bucket,key):
            print(f"Skip exists: s3://{bucket}/{key}")
            rows.append({"date":date,"rows":None,"key":key,"status":"skipped"})
            continue
        feature_df=build_trade_distribution_for_day(symbol,date,bucket,raw_prefix,interval,s3)
        buffer=io.BytesIO(); feature_df.to_parquet(buffer,index=False,engine="pyarrow",compression="zstd")
        s3.put_object(Bucket=bucket,Key=key,Body=buffer.getvalue())
        print(f"Uploaded: s3://{bucket}/{key} rows={len(feature_df)}")
        rows.append({"date":date,"rows":len(feature_df),"key":key,"status":"uploaded"})
    return pd.DataFrame(rows)

In [6]:
# ??????? ???????? ?? ????? ??? ????? ???????? ???????.
s3 = make_s3_client()
dates = list_symbol_trades_days(SYMBOL, s3_client=s3)
example_date = dates[1]
example_features = build_trade_distribution_for_day(SYMBOL, example_date, s3_client=s3)
print(example_date, example_features.shape)
display(example_features.head())
display(example_features.tail())
display(example_features.dtypes)

2020-02-02 (1440, 11)


,open_time,Cp_buy,Cp_sell,H_buy,H_sell,H_norm_buy,H_norm_sell,N_eff_buy,N_eff_sell,C_eff_buy,C_eff_sell
0,1580601600000,0.357638,0.695836,2.723967,0.836481,0.909283,0.603394,14.111519,1.860510,0.705576,0.465128
1,1580601660000,0.534666,0.642312,2.296686,2.308033,0.794599,0.736099,7.570540,6.197653,0.420586,0.269463
2,1580601720000,0.565934,0.426082,2.240829,2.758623,0.761038,0.879805,7.114382,13.294389,0.374441,0.578017
3,1580601780000,0.505022,0.508164,2.187122,2.577642,0.807637,0.822085,7.777951,10.106314,0.518530,0.439405
4,1580601840000,0.429586,0.583314,2.721396,2.870608,0.856309,0.835940,12.952977,13.414408,0.539707,0.432723


,open_time,Cp_buy,Cp_sell,H_buy,H_sell,H_norm_buy,H_norm_sell,N_eff_buy,N_eff_sell,C_eff_buy,C_eff_sell
1435,1580687700000,0.555095,0.434746,0.851020,2.446300,0.774632,0.882316,2.155057,10.059491,0.718352,0.628718
1436,1580687760000,0.413331,0.476062,1.378808,2.910357,0.769527,0.864301,3.409668,14.475831,0.568278,0.499167
1437,1580687820000,0.588014,0.440723,0.956244,1.235287,0.594148,0.891071,2.189091,3.080317,0.437818,0.770079
1438,1580687880000,0.599986,0.209119,2.059203,2.017394,0.742701,0.918156,5.899455,6.900602,0.368716,0.766734
1439,1580687940000,0.582869,0.492995,2.170618,2.324755,0.801543,0.858461,6.523105,8.684319,0.434874,0.578955


open_time        Int64
Cp_buy         float32
Cp_sell        float32
H_buy          float32
H_sell         float32
H_norm_buy     float32
H_norm_sell    float32
N_eff_buy      float32
N_eff_sell     float32
C_eff_buy      float32
C_eff_sell     float32
dtype: object

In [7]:
# ???????? ?????? ???? ??????? parquet-?????? ????????? ? S3.
# ??????????????, ????? ?????? ????? ????????? ?????? ????????.
# write_results = write_trade_distribution_for_symbol_to_s3(s3_client=s3)
# display(write_results)

In [8]:
# ?????? ?? ???? ?????? ?? config.yaml.
from config_loader import load_config

config = load_config("config.yaml")
config_symbols = config["symbols"]
config_interval = config["interval"]
config_start_date = pd.Timestamp(config["date_range"]["start"]).date()
config_end_date = pd.Timestamp(config["date_range"]["end"]).date()
config_bucket = config["storage"]["bucket"]
config_raw_prefix = config["storage"]["prefix"]

s3 = make_s3_client()
all_write_results = []

for symbol in config_symbols:
    available_dates = list_symbol_trades_days(
        symbol=symbol,
        bucket=config_bucket,
        raw_prefix=config_raw_prefix,
        s3_client=s3,
    )
    selected_dates = [
        date
        for date in available_dates
        if config_start_date <= pd.Timestamp(date).date() <= config_end_date
    ]

    if not selected_dates:
        print(f"No trades days found in config range for symbol={symbol}")
        continue

    rows = []
    for date in selected_dates:
        key = feature_dataset_key(symbol=symbol, date=date, interval=config_interval)

        if SKIP_EXISTING and s3_key_exists(s3, config_bucket, key):
            print(f"Skip exists: s3://{config_bucket}/{key}")
            rows.append({"symbol": symbol, "date": date, "rows": None, "key": key, "status": "skipped"})
            continue

        feature_df = build_trade_distribution_for_day(
            symbol=symbol,
            date=date,
            bucket=config_bucket,
            raw_prefix=config_raw_prefix,
            interval=config_interval,
            s3_client=s3,
        )

        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=config_bucket, Key=key, Body=buffer.getvalue())

        print(f"Uploaded: s3://{config_bucket}/{key} rows={len(feature_df)}")
        rows.append({"symbol": symbol, "date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

    all_write_results.append(pd.DataFrame(rows))

write_results_config_period = (
    pd.concat(all_write_results, ignore_index=True)
    if all_write_results
    else pd.DataFrame(columns=["symbol", "date", "rows", "key", "status"])
)

display(write_results_config_period)
print(write_results_config_period["status"].value_counts(dropna=False))

Uploaded: s3://binance-data-downloader/features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet rows=1439
Uploaded: s3://binance-data-downloader/features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-02/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-03/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-04/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-05/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-06/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/trade_distribution/symbol=ADAUSDT/interval=1m/date=2020-02-07/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/trade_distribution/sy

,symbol,date,rows,key,status
0,ADAUSDT,2020-02-01,1439,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
1,ADAUSDT,2020-02-02,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
2,ADAUSDT,2020-02-03,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
3,ADAUSDT,2020-02-04,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
4,ADAUSDT,2020-02-05,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
...,...,...,...,...,...
2188,ADAUSDT,2026-01-28,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
2189,ADAUSDT,2026-01-29,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
2190,ADAUSDT,2026-01-30,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded
2191,ADAUSDT,2026-01-31,1440,features/trade_distribution/symbol=ADAUSDT/int...,uploaded


status
uploaded    2193
Name: count, dtype: int64
